In [ ]:
# 1. Establish a connection between Python and the Sakila database.

from sqlalchemy import create_engine
import pandas as pd

# Rellena con tus datos reales
USER = "root"
PASSWORD = "xxxx"   # cambiar por contraseña de MySQL
HOST = "localhost"      # o la IP/host donde esté MySQL
PORT = "3306"           # puerto por defecto de MySQL
DB = "sakila"

# Cadena de conexión usando el driver pymysql
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}"

engine = create_engine(connection_string)


In [30]:
# 2. Write a Python function called rentals_month that retrieves rental data for a 
# given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame.
#  The function should take in three parameters

def rentals_month(engine, month, year):
    """
    Recupera todos los alquileres del mes y año especificados.
    """
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month}
          AND YEAR(rental_date) = {year};
    """
    df = pd.read_sql(query, engine)
    return df


In [32]:
# 3. Develop a Python function called rental_count_month that takes the DataFrame provided 
# by rentals_month as input along with the month and year and returns a new DataFrame containing 
# the number of rentals made by each customer_id during the selected month and year.


def rental_count_month(df, month, year):
    """
    Devuelve un DataFrame con el número total de alquileres por cliente.
    """
    col_name = f"alquileres_{month:02d}_{year}"
    df_counts = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=col_name)
    )
    return df_counts



In [33]:

# 4. Create a Python function called compare_rentals that takes two DataFrames as input containing 
# the number of rentals made by each customer in different months and years. 
# The function should return a combined DataFrame with a new 'difference' column, which is 
# the difference between the number of rentals in the two months.

def compare_rentals(df1, df2):
    """
    Combina dos DataFrames de conteo de alquileres por cliente
    y calcula la diferencia entre ambos meses.
    """
    df_merged = pd.merge(df1, df2, on="customer_id", how="outer").fillna(0)
    col1 = df1.columns[1]
    col2 = df2.columns[1]
    df_merged["diferencia"] = df_merged[col2] - df_merged[col1]
    return df_merged


In [34]:
#check

df_may = rentals_month(engine, 5, 2005)
df_jun = rentals_month(engine, 6, 2005)
count_may = rental_count_month(df_may, 5, 2005)
count_jun = rental_count_month(df_jun, 6, 2005)
comparison = compare_rentals(count_may, count_jun)
comparison.head()


,customer_id,alquileres_05_2005,alquileres_06_2005,diferencia
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0
